# `gold.bridge_ticker_manager` — load

Resolves the many-to-many. Grain is (ticker version, manager).

Keyed on the **versioned** `ticker_key`, so when a manager change opens a new version of a
trust the new version gets its own bridge rows and the old ones stay attached to the old
version. That is the one place the SCD2 history is still read.

`allocation_factor` is one over the number of managers on that version, so the weights sum
to one per trust. Ignore it and a trust counts once per manager — right for "what share of
managers beat the index". Apply it and the total still reconciles to the fact.

In [ ]:
CREATE OR REPLACE TEMP VIEW gold_stage_bridge_ticker_manager AS
WITH named AS (
  -- POSEXPLODE keeps the order the source listed the managers in. That is source order,
  -- not seniority, which is why nothing downstream treats position 1 as the lead.
  SELECT d.ticker_key, pos + 1 AS manager_position, TRIM(m) AS manager_name
  FROM `index-vs-trust-pipeline`.gold.dim_ticker d
  LATERAL VIEW POSEXPLODE(SPLIT(d.manager, ', ')) t AS pos, m
  WHERE d.manager NOT IN ('NoInfo', 'NotApplicable')
)
SELECT ticker_key,
       MD5(manager_name) AS manager_key,
       manager_position,
       1.0 / COUNT(*) OVER (PARTITION BY ticker_key) AS allocation_factor
FROM named;

In [ ]:
MERGE INTO `index-vs-trust-pipeline`.gold.bridge_ticker_manager AS t
USING gold_stage_bridge_ticker_manager AS s
   ON t.ticker_key = s.ticker_key AND t.manager_key = s.manager_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Verification

Expected: **230** rows over **97** tickers and **227** managers, **0** duplicate pairs, and
a longest manager list of **12** (NB Private Equity Partners, a fund of funds — the length
is real, not a parsing failure).

97, not 102: the three index rows and the two trusts with no named manager get no bridge
rows at all.

In [ ]:
SELECT COUNT(*)                          AS rows_total,
       COUNT(DISTINCT ticker_key)        AS tickers,
       COUNT(DISTINCT manager_key)       AS managers,
       COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', ticker_key, manager_key)) AS duplicate_pairs,
       MAX(manager_position)             AS longest_list
FROM `index-vs-trust-pipeline`.gold.bridge_ticker_manager;

Referential integrity and the weights. Expected: **0** orphans on either side, **0** trusts
whose weights miss 1.0, and the factors summing to **97** — one whole trust per trust.

In [ ]:
WITH per_ticker AS (
  SELECT ticker_key, SUM(allocation_factor) AS weight
  FROM `index-vs-trust-pipeline`.gold.bridge_ticker_manager
  GROUP BY ticker_key
)
SELECT (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.gold.bridge_ticker_manager b
         LEFT JOIN `index-vs-trust-pipeline`.gold.dim_ticker d ON d.ticker_key = b.ticker_key
        WHERE d.ticker_key IS NULL)                              AS orphan_ticker_keys,
       (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.gold.bridge_ticker_manager b
         LEFT JOIN `index-vs-trust-pipeline`.gold.dim_manager m ON m.manager_key = b.manager_key
        WHERE m.manager_key IS NULL)                             AS orphan_manager_keys,
       (SELECT COUNT(*) FROM per_ticker WHERE ABS(weight - 1.0) > 1e-9) AS weights_off,
       (SELECT ROUND(SUM(weight), 6) FROM per_ticker)            AS total_weight;

The three managers who make this a bridge rather than a foreign key. Expected: **two rows
each** — Sat Duhra on BNKR and HFEL, Simon Gergel on BUT and MRCH, Anthony Lynch on JCH
and MRC.

In [ ]:
SELECT m.manager_name, COUNT(*) AS bridge_rows, CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(d.ticker))) AS trusts
FROM `index-vs-trust-pipeline`.gold.bridge_ticker_manager b
JOIN `index-vs-trust-pipeline`.gold.dim_manager m ON m.manager_key = b.manager_key
JOIN `index-vs-trust-pipeline`.gold.dim_ticker  d ON d.ticker_key  = b.ticker_key
GROUP BY m.manager_name
HAVING COUNT(*) > 1
ORDER BY m.manager_name;